In [0]:
from pyspark.sql import functions as F
 
spark.sql("CREATE SCHEMA IF NOT EXISTS supply_chain_opt")
spark.sql("DROP TABLE IF EXISTS supply_chain_opt.inventory")
spark.sql("DROP TABLE IF EXISTS supply_chain_opt.vendor_logs")
 
NUM_PRODUCTS = 100000
NUM_VENDORS = 50

In [0]:
vendor_logs_df = spark.range(1, 100001).select(
    F.concat(F.lit("ORD_"), F.col("id")).alias("order_id"),
    F.concat(F.lit("VND_"), F.floor(F.rand() * NUM_VENDORS + 1)).alias("vendor_id"),
    F.date_add(F.to_date(F.lit("2025-01-01")), F.floor(F.rand() * 365).cast("int")).alias("order_date"),
    F.floor(F.rand() * 500 + 10).alias("quantity_ordered"),
    F.round(F.rand() * 0.4 + 0.6, 2).alias("quality_score")
)
 
vendor_logs_df = vendor_logs_df.withColumn(
    "delivery_date",
    F.date_add(F.col("order_date"), F.floor(F.rand() * 15 + 2).cast("int"))
)
 
vendor_logs_df.write.mode("overwrite").saveAsTable("supply_chain_opt.vendor_logs")
print(f"vendor_logs: {vendor_logs_df.count()} rows generated across {NUM_VENDORS} vendors.")
 

vendor_logs: 100000 rows generated across 50 vendors.


In [0]:
inventory_df = spark.range(1, NUM_PRODUCTS + 1).select(
    F.col("id").alias("product_id"),
    F.concat(F.lit("Item_"), F.col("id")).alias("product_name"),
    F.concat(F.lit("VND_"), F.floor(F.rand() * NUM_VENDORS + 1)).alias("vendor_id"),
    F.round(F.rand() * 45 + 5, 1).alias("daily_demand"),          # 5-50 units/day
    F.floor(F.rand() * 20 + 2).alias("lead_time_days"),           # 2-21 days
    F.round(F.rand() * 500 + 10, 2).alias("unit_cost"),
    F.round(F.rand() * 4 + 3, 1).alias("safety_stock_days")       # 3-7 days of buffer
)
 
inventory_df = inventory_df.withColumn(
    "reorder_point",
    F.round(
        F.col("daily_demand") * F.col("lead_time_days") + F.col("daily_demand") * F.col("safety_stock_days")
    ).cast("int")
).withColumn(
    # current_stock as a random fraction (0.1x - 1.8x) of reorder_point -> correlated, not independent.
    # This naturally produces a realistic mix of healthy and low-stock SKUs.
    "current_stock",
    F.round(F.col("reorder_point") * (F.rand() * 1.7 + 0.1)).cast("int")
)
 
inventory_df.write.mode("overwrite").saveAsTable("supply_chain_opt.inventory")
print(f"inventory: {inventory_df.count()} rows generated. reorder_point is now derived, not random.")

inventory: 100000 rows generated. reorder_point is now derived, not random.


In [0]:
display(spark.table("supply_chain_opt.inventory").limit(10))

product_id,product_name,vendor_id,daily_demand,lead_time_days,unit_cost,safety_stock_days,reorder_point,current_stock
25001,Item_25001,VND_27,43.3,7,410.49,6.3,576,1017
25002,Item_25002,VND_35,38.3,4,104.05,6.1,387,231
25003,Item_25003,VND_23,43.1,6,164.88,6.4,534,181
25004,Item_25004,VND_31,22.4,6,271.82,6.8,287,80
25005,Item_25005,VND_10,29.2,12,342.19,4.5,482,640
25006,Item_25006,VND_5,29.5,3,275.54,6.0,266,375
25007,Item_25007,VND_17,7.0,12,16.93,3.2,106,90
25008,Item_25008,VND_44,20.4,19,384.4,5.0,490,479
25009,Item_25009,VND_35,12.6,20,63.46,6.2,330,470
25010,Item_25010,VND_32,24.3,17,285.29,3.6,501,559


In [0]:
display(spark.table("supply_chain_opt.vendor_logs").limit(10))

order_id,vendor_id,order_date,quantity_ordered,quality_score,delivery_date
ORD_12501,VND_42,2025-12-23,122,0.71,2025-12-30
ORD_12502,VND_12,2025-08-30,38,0.61,2025-09-08
ORD_12503,VND_19,2025-11-14,469,0.86,2025-11-26
ORD_12504,VND_32,2025-05-22,312,0.74,2025-05-28
ORD_12505,VND_24,2025-12-31,206,0.9,2026-01-04
ORD_12506,VND_9,2025-04-08,204,0.67,2025-04-13
ORD_12507,VND_34,2025-03-31,182,0.74,2025-04-05
ORD_12508,VND_3,2025-07-23,134,0.95,2025-07-31
ORD_12509,VND_7,2025-04-16,22,0.9,2025-04-27
ORD_12510,VND_36,2025-04-19,406,0.79,2025-05-04


In [0]:
display(spark.sql("SELECT count(*) FROM supply_chain_opt.inventory"))

count(*)
100000


In [0]:
display(spark.sql("SELECT count(*) FROM supply_chain_opt.vendor_logs"))

count(*)
100000
